In [0]:
if spark.catalog.tableExists("oc_projeto.`oc-tabelas`.gold_af_dep"):
    max_ts = spark.sql("""
        SELECT max(data_processamento) as ts FROM oc_projeto.`oc-tabelas`.gold_af_dep
    """).collect()[0]["ts"]
else:
    max_ts = None



In [0]:
if max_ts is None:
    df_silver_af_dep = spark.table("oc_projeto.`oc-tabelas`.silver_voos_dep")
else:
    df_silver_af_dep = spark.table("oc_projeto.`oc-tabelas`.silver_voos_dep").filter(f"data_processamento > '{max_ts}'")



In [0]:
df_silver_af_dep.createOrReplaceTempView("temp_af_dep")

df_gold_af_dep = spark.sql("""
    SELECT
        id_voo as id_af_dep,
        data,
        voo_dep,
        iata_dep,
        af_origens_voo,
        af_origens_iata,
        af_dep,
        qtd as qtd_af_dep,
        turno,
        coalesce(qtd, 0) AS qtd_af,
        data_processamento
    FROM temp_af_dep            
""")

In [0]:
from pyspark.sql.functions import current_timestamp

df_gold_af_dep = df_gold_af_dep.withColumn("data_process_gold", current_timestamp())

In [0]:
if not spark.catalog.tableExists("oc_projeto.`oc-tabelas`.gold_af_dep"):
    df_gold_af_dep.write.format("delta").saveAsTable("oc_projeto.`oc-tabelas`.gold_af_dep")
else:
    df_gold_af_dep.createOrReplaceTempView("temp_gold_af_dep")

    spark.sql("""
          MERGE INTO oc_projeto.`oc-tabelas`.gold_af_dep AS t
          USING temp_gold_af_dep AS s
          ON t.id_af_dep = s.id_af_dep
          WHEN NOT MATCHED THEN INSERT *
          """)
print("Dados carregados com sucesso na gold_af_arr com MERGE")